<a href="https://colab.research.google.com/github/Udaykiran606/Zepto-AI-ML-Capstone/blob/main/analysis/Complete_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# -*- coding: utf-8 -*-
"""analytics.py

Module 2 — Analytics Pipeline (/analytics)
Fully integrated, end-to-end data profiling, cleaning, EDA, classification, tuning, and regression.
"""

import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import joblib

# Scikit-learn preprocessing & pipeline utilities
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV

# Scikit-learn estimators
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier

# Imbalanced-learn SMOTE
from imblearn.over_sampling import SMOTE

# Scikit-learn metrics
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

# Configure display options and visual aesthetics
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
sns.set_theme(style="whitegrid", palette="muted")

# ==============================================================================
# SECTION 1: DATA INGESTION & OFFLINE FALLBACK CHECK (EXACTLY ONCE)
# ==============================================================================
print("=" * 70)
print("SECTION 1: DATA INGESTION & OFFLINE FALLBACK CHECK")
print("=" * 70)

os.makedirs("analytics", exist_ok=True)
csv_path = os.path.join("analytics", "titanic.csv")

# Load raw dataset once from Seaborn cache/network or offline fallback
try:
    df_raw = sns.load_dataset("titanic")
    df_raw.to_csv(csv_path, index=False)
    print(f"✅ Loaded raw dataset from Seaborn and saved fallback artifact: '{csv_path}'.")
except Exception:
    if os.path.exists(csv_path):
        df_raw = pd.read_csv(csv_path)
        print(f"✅ Loaded raw dataset from committed offline fallback: '{csv_path}'.")
    else:
        raise RuntimeError("Could not load dataset from network and no offline fallback found.")

print("\n--- Raw Dataset Profiling ---")
print(f"Shape: {df_raw.shape}")
print("\n--- Info ---")
df_raw.info()
print("\n--- Summary Statistics ---")
print(df_raw.describe(include="all"))

# ==============================================================================
# SECTION 2: DEFENSIBLE DATA CLEANING & THRESHOLD RULES
# ==============================================================================
print("\n" + "=" * 70)
print("SECTION 2: DEFENSIBLE DATA CLEANING")
print("=" * 70)

missing_counts = df_raw.isnull().sum()
missing_pct = (missing_counts / len(df_raw)) * 100
affected = pd.DataFrame(
    {
        "Missing Count": missing_counts[missing_counts > 0],
        "Pct (%)": missing_pct[missing_pct > 0].round(2),
    }
).sort_values(by="Pct (%)", ascending=False)

print("Affected Missing Values (Pre-Cleaning):\n", affected)

df_clean = df_raw.copy()

# Rule A: High missingness (> 30%) -> Drop column (deck ~77.10% missing)
if "deck" in df_clean.columns:
    df_clean.drop(columns=["deck"], inplace=True)

# Rule B: 5%–30% missing range -> Grouped median imputation (age ~19.87% missing)
df_clean["age"] = df_clean.groupby(["pclass", "sex"])["age"].transform(
    lambda x: x.fillna(x.median())
)

# Rule C: Under 5% missingness -> Drop rows (embarked / embark_town ~0.22% missing)
subset_cols = [c for c in ["embarked", "embark_town"] if c in df_clean.columns]
df_clean.dropna(subset=subset_cols, inplace=True)

# Feature engineering
df_clean["family_size"] = df_clean["sibsp"] + df_clean["parch"] + 1

# Overwrite committed CSV with cleaned data artifact
df_clean.to_csv(csv_path, index=False)
print(f"\n✅ Cleaned dataset shape: {df_clean.shape}")

# ==============================================================================
# SECTION 3: UNIVARIATE ANALYSIS (IQR OUTLIERS & SKEWNESS)
# ==============================================================================
print("\n" + "=" * 70)
print("SECTION 3: UNIVARIATE ANALYSIS")
print("=" * 70)

def compute_iqr_outliers(series, name):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lb, ub = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outliers = series[(series < lb) | (series > ub)]
    print(f"{name} IQR Outliers: {len(outliers)} rows outside [{lb:.2f}, {ub:.2f}]")

compute_iqr_outliers(df_clean["age"], "Age")
compute_iqr_outliers(df_clean["fare"], "Fare")

fare_mean, fare_med, fare_mode = df_clean["fare"].mean(), df_clean["fare"].median(), df_clean["fare"].mode()[0]
print(f"\nFare Skewness check -> Mean: ${fare_mean:.2f} | Median: ${fare_med:.2f} | Mode: ${fare_mode:.2f}")
print("Interpretation: Since Mean ($32.20) > Median ($14.45) > Mode ($8.05), ticket fare is strongly RIGHT-SKEWED.")

# Histograms & Box Plots
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
sns.histplot(df_clean["age"], kde=True, ax=axes[0, 0], color="skyblue").set_title("Age Distribution")
sns.boxplot(x=df_clean["age"], ax=axes[0, 1], color="lightgreen").set_title("Age Box Plot")
sns.histplot(df_clean["fare"], kde=True, ax=axes[1, 0], color="salmon").set_title("Fare Distribution")
sns.boxplot(x=df_clean["fare"], ax=axes[1, 1], color="orchid").set_title("Fare Box Plot")
plt.tight_layout()
plt.savefig("analytics/univariate_plots.png")
plt.close()

# ==============================================================================
# SECTION 4: BIVARIATE ANALYSIS (BOOLEAN MASKING & 6x6 HEATMAP)
# ==============================================================================
print("\n" + "=" * 70)
print("SECTION 4: BIVARIATE ANALYSIS")
print("=" * 70)

# (a) Sex
for sex in df_clean["sex"].unique():
    print(f"Survival Rate - Sex ({sex}): {df_clean[df_clean['sex'] == sex]['survived'].mean():.2%}")

# (b) Pclass
for pclass in sorted(df_clean["pclass"].unique()):
    print(f"Survival Rate - Pclass ({pclass}): {df_clean[df_clean['pclass'] == pclass]['survived'].mean():.2%}")

# (c) Sex & Pclass
for sex in ["female", "male"]:
    for pclass in sorted(df_clean["pclass"].unique()):
        mask = (df_clean["sex"] == sex) & (df_clean["pclass"] == pclass)
        print(f"Survival Rate - Sex ({sex}) & Pclass ({pclass}): {df_clean[mask]['survived'].mean():.2%}")

# 6x6 Correlation Matrix (strictly specified numeric columns)
numeric_cols = ["survived", "pclass", "age", "sibsp", "parch", "fare"]
corr_matrix = df_clean[numeric_cols].corr()

# Extract top two off-diagonal correlations
corr_unstacked = corr_matrix.abs().unstack()
off_diag = corr_unstacked[corr_unstacked.index.get_level_values(0) != corr_unstacked.index.get_level_values(1)]
top_pairs = off_diag.drop_duplicates().nlargest(2)

print("\n--- Top 2 Strongest Off-Diagonal Correlations ---")
for (f1, f2), val in top_pairs.items():
    raw_corr = corr_matrix.loc[f1, f2]
    print(f"Pair: ({f1}, {f2}) -> Pearson Correlation = {raw_corr:.4f} (Abs = {val:.4f})")

plt.figure(figsize=(7, 5))
sns.heatmap(corr_matrix, annot=True, fmt=".3f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("6x6 Correlation Matrix")
plt.tight_layout()
plt.savefig("analytics/correlation_heatmap.png")
plt.close()

# ==============================================================================
# SECTION 5: MULTIVARIATE DATA STORY & EDA STANDARDIZATION CHECK
# ==============================================================================
print("\n" + "=" * 70)
print("SECTION 5: MULTIVARIATE DATA STORY & EDA SANITY CHECK")
print("=" * 70)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
sns.barplot(data=df_clean, x="pclass", y="survived", hue="sex", errorbar=None, ax=axes[0, 0]).set_title("1. Survival by Class & Sex")
sns.boxplot(data=df_clean, x="pclass", y="fare", hue="survived", ax=axes[0, 1]).set_yscale("log")
axes[0, 1].set_title("2. Log Fare by Class & Survival")
sns.scatterplot(data=df_clean, x="age", y="fare", hue="survived", style="sex", alpha=0.7, ax=axes[1, 0]).set_title("3. Age vs Fare by Survival & Sex")
sns.lineplot(data=df_clean, x="family_size", y="survived", hue="sex", marker="o", errorbar=None, ax=axes[1, 1]).set_title("4. Survival Rate by Family Size & Sex")
plt.tight_layout()
plt.savefig("analytics/multivariate_story.png")
plt.close()

# EDA Standardization Sanity Check
eda_scaler = StandardScaler()
scaled_vals = eda_scaler.fit_transform(df_clean[["age", "fare"]])
print("EDA Standardization Check Summary:")
print(pd.DataFrame({
    "Feature": ["age", "fare"],
    "Raw Mean": [df_clean["age"].mean(), df_clean["fare"].mean()],
    "Scaled Mean": [scaled_vals[:, 0].mean(), scaled_vals[:, 1].mean()],
    "Raw Std": [df_clean["age"].std(), df_clean["fare"].std()],
    "Scaled Std": [scaled_vals[:, 0].std(), scaled_vals[:, 1].std()]
}))

# ==============================================================================
# SECTION 6: STRATIFIED SPLIT & LEAKAGE-FREE PIPELINE SETUP
# ==============================================================================
print("\n" + "=" * 70)
print("SECTION 6: STRATIFIED SPLIT & LEAKAGE-FREE PIPELINE SETUP")
print("=" * 70)

target_col = "survived"
cols_to_drop = [c for c in ["survived", "alive", "class", "embarked"] if c in df_clean.columns]
feature_cols = [c for c in df_clean.columns if c not in cols_to_drop]

X = df_clean[feature_cols].copy()
y = df_clean[target_col].copy()

# Stratified Train/Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Train Size: {len(X_train)} ({y_train.mean():.2%} survivors)")
print(f"Test Size : {len(X_test)} ({y_test.mean():.2%} survivors)")

# ColumnTransformer Setup
numeric_features = ["age", "fare", "sibsp", "parch", "family_size"]
categorical_features = ["sex", "pclass", "embark_town", "who", "adult_male", "alone"]

num_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
cat_pipeline = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore", drop="first"))])

preprocessor = ColumnTransformer(transformers=[("num", num_pipeline, numeric_features), ("cat", cat_pipeline, categorical_features)])

# ==============================================================================
# SECTION 7: THREE CLASSIFIERS & EVALUATION
# ==============================================================================
print("\n" + "=" * 70)
print("SECTION 7: THREE CLASSIFIERS & EVALUATION")
print("=" * 70)

classifiers = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=3, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

clf_results = {}
fitted_pipelines = {}

plt.figure(figsize=(8, 5))
for name, clf in classifiers.items():
    pipe = Pipeline([("preprocessor", preprocessor), ("classifier", clf)])
    pipe.fit(X_train, y_train)
    fitted_pipelines[name] = pipe

    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]

    cm = confusion_matrix(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc_val = roc_auc_score(y_test, y_prob)

    clf_results[name] = {
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1 Score": round(f1, 4),
        "ROC AUC": round(auc_val, 4),
        "Confusion Matrix": cm.ravel().tolist()
    }

    fpr, tpr, _ = roc_curve(y_test, y_prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc_val:.3f})")

plt.plot([0, 1], [0, 1], "k--")
plt.title("ROC Curves Comparison")
plt.legend()
plt.tight_layout()
plt.savefig("analytics/roc_curves.png")
plt.close()

df_clf_metrics = pd.DataFrame(clf_results).T
print("Classifiers Performance Summary:")
print(df_clf_metrics)

# Render Decision Tree
dt_pipe = fitted_pipelines["Decision Tree"]
ohe_names = list(dt_pipe.named_steps["preprocessor"].named_transformers_["cat"].named_steps["onehot"].get_feature_names_out(categorical_features))
all_feat_names = numeric_features + ohe_names

plt.figure(figsize=(16, 8))
plot_tree(dt_pipe.named_steps["classifier"], feature_names=all_feat_names, class_names=["Perished", "Survived"], filled=True, rounded=True)
plt.title("Decision Tree Visualization")
plt.tight_layout()
plt.savefig("analytics/decision_tree.png")
plt.close()

# ==============================================================================
# SECTION 8: CLASS IMBALANCE HANDLING COMPARISON (WITH SMOTE)
# ==============================================================================
print("\n" + "=" * 70)
print("SECTION 8: CLASS IMBALANCE EXPERIMENT")
print("=" * 70)

# (a) Baseline
pipe_base = Pipeline([("preprocessor", preprocessor), ("classifier", LogisticRegression(random_state=42))])
pipe_base.fit(X_train, y_train)
y_p_base = pipe_base.predict(X_test)

# (b) Class Weight Balanced
pipe_bal = Pipeline([("preprocessor", preprocessor), ("classifier", LogisticRegression(class_weight="balanced", random_state=42))])
pipe_bal.fit(X_train, y_train)
y_p_bal = pipe_bal.predict(X_test)

# (c) SMOTE Oversampling (applied ONLY on training fold)
X_tr_proc = preprocessor.fit_transform(X_train)
X_te_proc = preprocessor.transform(X_test)

smote = SMOTE(random_state=42)
X_tr_smote, y_tr_smote = smote.fit_resample(X_tr_proc, y_train)

clf_smote = LogisticRegression(random_state=42)
clf_smote.fit(X_tr_smote, y_tr_smote)
y_p_smote = clf_smote.predict(X_te_proc)

imb_results = [
    {"Strategy": "Baseline", "Precision": round(precision_score(y_test, y_p_base), 4), "Recall": round(recall_score(y_test, y_p_base), 4), "F1 Score": round(f1_score(y_test, y_p_base), 4)},
    {"Strategy": "Class Weight Balanced", "Precision": round(precision_score(y_test, y_p_bal), 4), "Recall": round(recall_score(y_test, y_p_bal), 4), "F1 Score": round(f1_score(y_test, y_p_bal), 4)},
    {"Strategy": "SMOTE Oversampling (Train-Only)", "Precision": round(precision_score(y_test, y_p_smote), 4), "Recall": round(recall_score(y_test, y_p_smote), 4), "F1 Score": round(f1_score(y_test, y_p_smote), 4)}
]

print(pd.DataFrame(imb_results).to_string(index=False))

# ==============================================================================
# SECTION 9: GRIDSEARCHCV TUNING & OOB SCORE EVALUATION
# ==============================================================================
print("\n" + "=" * 70)
print("SECTION 9: GRIDSEARCHCV TUNING & OOB EVALUATION")
print("=" * 70)

tuning_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(oob_score=True, random_state=42))
])

param_grid = {
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [4, 6, 8],
    "classifier__max_features": ["sqrt", "log2"]
}

grid_search = GridSearchCV(estimator=tuning_pipeline, param_grid=param_grid, cv=5, scoring="roc_auc", n_jobs=-1)
grid_search.fit(X_train, y_train)

best_rf_pipe = grid_search.best_estimator_
oob_val = best_rf_pipe.named_steps["classifier"].oob_score_

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Out-Of-Bag (OOB) Score: {oob_val:.4f} ({oob_val:.2%})")

y_pred_tuned = best_rf_pipe.predict(X_test)
y_prob_tuned = best_rf_pipe.predict_proba(X_test)[:, 1]

clf_results["Tuned Random Forest"] = {
    "Accuracy": round(accuracy_score(y_test, y_pred_tuned), 4),
    "Precision": round(precision_score(y_test, y_pred_tuned), 4),
    "Recall": round(recall_score(y_test, y_pred_tuned), 4),
    "F1 Score": round(f1_score(y_test, y_pred_tuned), 4),
    "ROC AUC": round(roc_auc_score(y_test, y_prob_tuned), 4),
    "Confusion Matrix": confusion_matrix(y_test, y_pred_tuned).ravel().tolist()
}

# ==============================================================================
# SECTION 10: REGRESSION SIDE-TASK (PREDICTING FARE) & RESIDUAL ANALYSIS
# ==============================================================================
print("\n" + "=" * 70)
print("SECTION 10: REGRESSION SIDE-TASK (MULTIVARIATE LINEAR REGRESSION)")
print("=" * 70)

drop_reg_cols = [c for c in ["fare", "alive", "class", "embarked"] if c in df_clean.columns]
X_reg = df_clean.drop(columns=drop_reg_cols)
y_reg = df_clean["fare"]

reg_num_cols = ["age", "sibsp", "parch", "family_size"]
reg_cat_cols = ["sex", "pclass", "embark_town", "who", "adult_male", "alone"]

reg_preprocessor = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), reg_num_cols),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore", drop="first"))]), reg_cat_cols)
])

X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(X_reg, y_reg, test_size=0.20, random_state=42)

lin_reg_pipe = Pipeline([
    ("preprocessor", reg_preprocessor),
    ("regressor", LinearRegression())
])
lin_reg_pipe.fit(X_tr_r, y_tr_r)
reg_preds = lin_reg_pipe.predict(X_te_r)

mae_val = mean_absolute_error(y_te_r, reg_preds)
mse_val = mean_squared_error(y_te_r, reg_preds)
rmse_val = np.sqrt(mse_val)
r2_val = r2_score(y_te_r, reg_preds)

n_samples_reg = len(y_te_r)
k_features_reg = reg_preprocessor.transform(X_tr_r).shape[1]
adj_r2_val = 1 - ((1 - r2_val) * (n_samples_reg - 1) / (n_samples_reg - k_features_reg - 1))

print(f"MAE        : {mae_val:.4f}")
print(f"RMSE       : {rmse_val:.4f}")
print(f"R²         : {r2_val:.4f}")
print(f"Adjusted R²: {adj_r2_val:.4f}")

# Residual Plot Generation
residuals = y_te_r - reg_preds

plt.figure(figsize=(8, 5))
plt.scatter(reg_preds, residuals, alpha=0.6, color="purple", edgecolors="k")
plt.axhline(0, color="red", linestyle="--", linewidth=1.5)
plt.title("Residuals vs Predicted Fare (Multivariate Linear Regression)")
plt.xlabel("Predicted Fare ($)")
plt.ylabel("Residuals (Actual - Predicted)")
plt.tight_layout()
plt.savefig("analytics/fare_regression_residuals.png")
plt.close()

# ==============================================================================
# SECTION 11: MODEL COMPARISON TABLE & RECOMMENDATION
# ==============================================================================
print("\n" + "=" * 70)
print("SECTION 11: MODEL COMPARISON TABLE & RECOMMENDATION")
print("=" * 70)

comparison_table = pd.DataFrame([
    {
        "Task / Model": "Classification - Logistic Regression",
        "Accuracy": clf_results["Logistic Regression"]["Accuracy"],
        "Precision": clf_results["Logistic Regression"]["Precision"],
        "Recall": clf_results["Logistic Regression"]["Recall"],
        "F1 Score": clf_results["Logistic Regression"]["F1 Score"],
        "ROC AUC": clf_results["Logistic Regression"]["ROC AUC"],
        "MAE": "-", "RMSE": "-", "R²": "-", "Adjusted R²": "-"
    },
    {
        "Task / Model": "Classification - Decision Tree",
        "Accuracy": clf_results["Decision Tree"]["Accuracy"],
        "Precision": clf_results["Decision Tree"]["Precision"],
        "Recall": clf_results["Decision Tree"]["Recall"],
        "F1 Score": clf_results["Decision Tree"]["F1 Score"],
        "ROC AUC": clf_results["Decision Tree"]["ROC AUC"],
        "MAE": "-", "RMSE": "-", "R²": "-", "Adjusted R²": "-"
    },
    {
        "Task / Model": "Classification - Random Forest (Default)",
        "Accuracy": clf_results["Random Forest"]["Accuracy"],
        "Precision": clf_results["Random Forest"]["Precision"],
        "Recall": clf_results["Random Forest"]["Recall"],
        "F1 Score": clf_results["Random Forest"]["F1 Score"],
        "ROC AUC": clf_results["Random Forest"]["ROC AUC"],
        "MAE": "-", "RMSE": "-", "R²": "-", "Adjusted R²": "-"
    },
    {
        "Task / Model": "Classification - Tuned Random Forest",
        "Accuracy": clf_results["Tuned Random Forest"]["Accuracy"],
        "Precision": clf_results["Tuned Random Forest"]["Precision"],
        "Recall": clf_results["Tuned Random Forest"]["Recall"],
        "F1 Score": clf_results["Tuned Random Forest"]["F1 Score"],
        "ROC AUC": clf_results["Tuned Random Forest"]["ROC AUC"],
        "MAE": "-", "RMSE": "-", "R²": "-", "Adjusted R²": "-"
    },
    {
        "Task / Model": "Regression - Multivariate Linear Regression",
        "Accuracy": "-", "Precision": "-", "Recall": "-", "F1 Score": "-", "ROC AUC": "-",
        "MAE": round(mae_val, 4), "RMSE": round(rmse_val, 4), "R²": round(r2_val, 4), "Adjusted R²": round(adj_r2_val, 4)
    }
])

print("\n--- Final Model Comparison Table ---")
print(comparison_table.to_string(index=False))

# ==============================================================================
# SECTION 12: MODEL SERIALIZATION & END-TO-END INFERENCE CONFIRMATION
# ==============================================================================
print("\n" + "=" * 70)
print("SECTION 12: MODEL SERIALIZATION & END-TO-END VERIFICATION")
print("=" * 70)

model_artifact_path = os.path.join("analytics", "fitted_pipeline.joblib")
joblib.dump(best_rf_pipe, model_artifact_path)
print(f"✅ Complete fitted pipeline successfully saved to '{model_artifact_path}'.")

# End-to-end inference verification
reloaded_pipeline = joblib.load(model_artifact_path)

raw_input_data = pd.DataFrame([
    {"pclass": 1, "sex": "female", "age": 28.0, "sibsp": 0, "parch": 0, "fare": 85.50, "embark_town": "Cherbourg", "who": "woman", "adult_male": False, "alone": True, "family_size": 1},
    {"pclass": 3, "sex": "male", "age": 22.0, "sibsp": 1, "parch": 0, "fare": 7.25, "embark_town": "Southampton", "who": "man", "adult_male": True, "alone": False, "family_size": 2}
])

raw_predictions = reloaded_pipeline.predict(raw_input_data)
raw_probabilities = reloaded_pipeline.predict_proba(raw_input_data)[:, 1]

print("\n--- End-to-End Prediction Verification ---")
for i, (pred, prob) in enumerate(zip(raw_predictions, raw_probabilities)):
    status = "Survived" if pred == 1 else "Perished"
    print(f"Sample {i+1}: Output = {status:<8} | Survival Probability = {prob:.4f}")

print("\n✅ Verification complete! The analytics pipeline is fully operational.")

SECTION 1: DATA INGESTION & OFFLINE FALLBACK CHECK
✅ Loaded raw dataset from Seaborn and saved fallback artifact: 'analytics/titanic.csv'.

--- Raw Dataset Profiling ---
Shape: (891, 15)

--- Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    object  
 13  alive        891 non-null    object  
 14  a